# AIMO3 Inference Scaling Demo

This notebook demonstrates the entropy-weighted voting approach for mathematical reasoning.

**Note:** Full inference requires H100 80GB GPU. This demo shows the voting logic only.

In [ ]:
# Install dependencies
!pip install -q numpy

In [ ]:
import numpy as np
from collections import defaultdict
from math import log, sqrt

## 5-Component Weighted Entropy

Our key finding: position-weighted entropy (40% weight) significantly outperforms simple averaging.

In [ ]:
def compute_weighted_entropy(logprobs):
    """
    5-component weighted entropy calculation:
    1. Mean entropy (30%)
    2. Variance penalty (20%)
    3. Position-weighted entropy (40%) - emphasizes final tokens
    4. High-entropy penalty
    5. Low-entropy streak bonus
    """
    if not logprobs:
        return float('inf')
    
    # Calculate per-token entropy
    entropies = []
    for probs in logprobs:
        # Shannon entropy
        ent = -sum(p * log(p + 1e-10) for p in probs if p > 0)
        entropies.append(ent)
    
    n = len(entropies)
    if n == 0:
        return float('inf')
    
    # 1. Mean entropy (30%)
    mean_ent = sum(entropies) / n
    
    # 2. Variance penalty (20%)
    variance = sum((e - mean_ent)**2 for e in entropies) / n
    
    # 3. Position-weighted entropy (40%) - exponential decay emphasizing end
    decay = 0.995
    weights = [decay ** (n - i - 1) for i in range(n)]
    position_weighted = sum(e * w for e, w in zip(entropies, weights)) / sum(weights)
    
    # 4. High-entropy penalty
    high_ent_ratio = sum(1 for e in entropies if e > 2.0) / n
    
    # 5. Combine components
    weighted_entropy = (
        0.3 * mean_ent +
        0.4 * position_weighted +
        0.2 * sqrt(variance) +
        0.3 * high_ent_ratio * 3.0
    )
    
    return weighted_entropy

In [ ]:
def select_answer(results):
    """
    Select answer with highest confidence-weighted votes.
    
    Args:
        results: List of dicts with 'answer' and 'logprobs' keys
    
    Returns:
        Selected answer with highest weighted votes
    """
    ans_weights = defaultdict(float)
    
    for r in results:
        if r.get('answer') is not None:
            entropy = compute_weighted_entropy(r.get('logprobs', []))
            weight = 1.0 / max(entropy, 1e-9)
            ans_weights[r['answer']] += weight
    
    if not ans_weights:
        return None
    
    return max(ans_weights.items(), key=lambda x: x[1])

## Demo: Simulated Voting

Demonstrate how entropy-weighted voting selects high-confidence answers.

In [ ]:
# Simulated results from 8 parallel attempts
# Low entropy = high confidence, High entropy = uncertain

simulated_results = [
    {'answer': 42, 'logprobs': [[0.95, 0.03, 0.02]] * 100},  # Very confident
    {'answer': 42, 'logprobs': [[0.90, 0.05, 0.05]] * 100},  # Confident
    {'answer': 42, 'logprobs': [[0.85, 0.10, 0.05]] * 100},  # Fairly confident
    {'answer': 37, 'logprobs': [[0.50, 0.30, 0.20]] * 100},  # Uncertain
    {'answer': 37, 'logprobs': [[0.40, 0.35, 0.25]] * 100},  # Very uncertain
    {'answer': 42, 'logprobs': [[0.88, 0.07, 0.05]] * 100},  # Confident
    {'answer': 99, 'logprobs': [[0.33, 0.33, 0.34]] * 100},  # Maximum uncertainty
    {'answer': 42, 'logprobs': [[0.92, 0.05, 0.03]] * 100},  # Very confident
]

# Run voting
winner, weight = select_answer(simulated_results)

print(f"Selected Answer: {winner}")
print(f"Total Weight: {weight:.2f}")
print()

# Show all answer weights
ans_weights = defaultdict(float)
for r in simulated_results:
    entropy = compute_weighted_entropy(r.get('logprobs', []))
    weight = 1.0 / max(entropy, 1e-9)
    ans_weights[r['answer']] += weight
    print(f"Answer {r['answer']}: entropy={entropy:.4f}, weight={weight:.2f}")

print()
print("Final Weights by Answer:")
for ans, w in sorted(ans_weights.items(), key=lambda x: -x[1]):
    print(f"  {ans}: {w:.2f}")

## Key Insight

Even though answer 37 has 2 votes vs answer 42's 5 votes, the entropy-weighted system strongly favors 42 because:
- 42's votes come from high-confidence (low entropy) solutions
- 37's votes come from uncertain (high entropy) solutions

This is why weighted entropy voting gained us +6 points over simple majority voting.

## Configuration Reference

Best configuration (V40 - 42/50):

In [ ]:
config = {
    # Model
    'model': 'danielhanchen/gpt-oss-120b',
    'context_tokens': 65536,
    'kv_cache_dtype': 'fp8_e4m3',
    
    # Sampling
    'temperature': 1.0,
    'min_p': 0.02,
    
    # Voting
    'attempts': 8,
    'early_stop': 4,
    'voting': 'weighted_entropy',
}

for k, v in config.items():
    print(f"{k}: {v}")

## Links

- [Kaggle Model](https://www.kaggle.com/models/danielhanchen/gpt-oss-120b)
- [HuggingFace Model](https://huggingface.co/unsloth/gpt-oss-120B)
- [GitHub Repo](https://github.com/PawanRamaMali/aimo3-inference-scaling)